In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import lora_transfer_pruning
from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [ ]:
    import importlib
    # importlib.reload(PruningInstrumentor)

In [3]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
DEVICE = "cuda:0"
def load_model(device=DEVICE):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        #quantization_config=quantization_config,
        #dtype=torch.bfloat16,
        device_map=device,
        # cache_dir="/glazkov-dev/.cache",
    )
    return model

In [ ]:
model = load_model()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [6]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(128256, 4096)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): LlamaRotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): BlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): LlamaDecoderLayer(
        (self_attn): PositionEmbeddingsAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.

https://github.com/VainF/Torch-Pruning?tab=readme-ov-file#sparse-training-optional

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [8]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [9]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [10]:
model.train()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): EmbeddingBridge(
      (hook_in): HookPoint(name='embed.hook_in')
      (hook_out): HookPoint(name='embed.hook_out')
      (_original_component): Embedding(128256, 4096)
    )
    (layers): ModuleList(
      (0): BlockBridge(
        (hook_in): HookPoint(name='blocks.0.hook_in')
        (hook_out): HookPoint(name='blocks.0.hook_out')
        (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
        (_original_component): LlamaDecoderLayer(
          (self_attn): PositionEmbeddingsAttentionBridge(
            (hook_in): HookPoint(name='blocks.0.attn.hook_in')
            (hook_out): HookPoint(name='blocks.0.attn.hook_out')
            (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
            (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
            (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
            (hook_result): HookPoint(name='blocks.0.attn.hook_resu

We don't want norms like LlamaRMSNorm(weight) with x/std(x)*weight being pruned.

In case of "local" pruning without residual stream we dont need it because in Llama all RMSNorms located after residual stream, so it have size of hidden_size that don't change and don't pruned.

In [11]:
ignored_params = []
for name, param in model.named_parameters():
    if "norm" in name:
        ignored_params.append(param)

In [12]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input=input_ids,
        use_cache=False,
        return_type="logits",
        #return_dict=True,
    ) #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    bridge,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
    #unwrapped_parameters=list(zip(ignored_params, [_] * len(ignored_params)))
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.9._original_component.input_layernorm._original_component.weight', 'model.layers.14._original_component.input_layernorm._original_component.weight', 'model.layers.14._original_component.post_attention_layernorm._original_component.weight', 'model.layers.19._original_component.input_layernorm._original_component.weight', 'model.layers.19._original_component.post_attention_layernorm._original_component.weight', 'model.layers.12._original_component.post_attention_layernorm._original_component.weight', 'model.layers.24._original_component.input_layernorm._original_component.weight', 'model.layers.30._original_component.input_layernorm._original_component.weight', 'model.layers.27._original_component.input_layernorm._original_component.weight', 'model.layers.27._original_component.post_attention_layernorm._original_component.

In [13]:
model.config.hidden_size

4096

LinearBridge(4096 -> 4096, bias=False, original_component=Linear)

In [ ]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )

In [ ]:
print("if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.")

if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.


~~unwrapped_parameters - parameters that is not in registered model.parameters()~~
unwrapped_parameters - parameters that just nn.Parameter like weight = nn.Parameter(torch.ones(5)) instead of nn.Linear()

!Torch pruning doesnt understand semantics of channels/rows of Parameter


Note: parameters setted via self.linear = nn.Linear() through `__setattr__`  
it should be nn.Parameter()

Can afford to find pruning groups only on primitive modules like nn.Linear that directly participate in computational graph, not LlamaMLP or composite layers.

But pruning in_channels doesn't have fanout effect

In [ ]:
from torch_pruning.dependency.constants import MAX_VALID_DIM
print(MAX_VALID_DIM)

18446744073709551616


In [ ]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )

In [ ]:
from torch_pruning.dependency.node import Node
q_proj_id = id(group[0][0].source) 
group[0]

(prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), [2, 6, 9])

rotary embedding operations:

In [ ]:
node_by_id = {
    id(node): node
    for node in DG.module2node.values()
}


In [ ]:
node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[0].outputs

[<Node: (_ConcatOp_1550(None))>]

In [ ]:
node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[1].outputs

[<Node: (_ElementWiseOp_1551(NegBackward0))>]

In [ ]:
#concat_op
node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[1].outputs[0].outputs

[<Node: (_ConcatOp_1550(None))>]

In [ ]:
concat_node = node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[1].outputs[0].outputs[0]

In [ ]:
concat_node.module

_ConcatOp_1550(None)

In [ ]:
concat_node.dependencies

[prune_out_channels on _ConcatOp_1550(None) => prune_out_channels on _ElementWiseOp_1551(NegBackward0),
 prune_out_channels on _ConcatOp_1550(None) => prune_out_channels on _Slice_1552(),
 prune_out_channels on _ConcatOp_1550(None) => prune_out_channels on _ElementWiseOp_1549(MulBackward0)]

In [ ]:
concat_node.outputs[0]

<Node: (_ElementWiseOp_1549(MulBackward0))>

In [ ]:
print(
    concat_node.grad_fn,
    getattr(concat_node.grad_fn, "_saved_dim", "missing"),
)

<CatBackward0 object at 0x7d375c356830> 18446744073709551615


In [ ]:
grad_fn = concat_node.grad_fn

print("type:", type(grad_fn))
print("name:", grad_fn.name())
print("next_functions:", grad_fn.next_functions)
print("metadata:", grad_fn.metadata)

type: <class 'CatBackward0'>
name: CatBackward0
next_functions: ((<NegBackward0 object at 0x7d375c356920>, 0), (<SliceBackward0 object at 0x7d375c3569b0>, 0))
metadata: {}


In [ ]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.5.1",
  "use_cache": true,
  "vocab_size": 128256
}

In [ ]:
bridge.blocks[0].attn.q._original_component

Linear(in_features=4096, out_features=4096, bias=False)

In [ ]:
bridge.blocks[0].attn.k._original_component

Linear(in_features=4096, out_features=1024, bias=False)

In [ ]:
bridge.blocks[0].attn.v._original_component

Linear(in_features=4096, out_features=1024, bias=False)

That means, that concat operation - about k or v _ConcatOp_1536([0, 1024, 2048])

In [ ]:
model.config.num_attention_heads * model.config.head_dim #out size of q

4096

32 * 128 = 4096 for llama

In [ ]:
tp.pruner.BasePruner #num_heads - mapping for every module q_proj, k_proj, v_proj that shows size of head in it's dim
tp.pruner.MetaPruner
#prune_head_dims
#prune_num_heads
#NOTE for prune_head_dims and prune_num_heads we need "num_heads" set

torch_pruning.pruner.algorithms.base_pruner.BasePruner

In [ ]:
print("prune q_proj group")
print(group)

prune q_proj group

--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=3
[1] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _Reshape_1554(), len(idxs)=3
[2] prune_out_channels on _Reshape_1554() => prune_out_channels on _ElementWiseOp_1553(TransposeBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prune_out_channels on _Slice_1552(), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_1553(TransposeBac

NOTE: tp.prune_multihead_attention_out_channels only for  nn.MultiheadAttention

In [ ]:
#tp.prune_multihead_attention_out_channels only for  nn.MultiheadAttention
#group2 = DG.get_pruning_group(
    # bridge.blocks[0].attn.q._original_component, 
    # tp.prune_multihead_attention_out_channels, 
    # idxs=[2, 6, 9] )

In [ ]:
def manually_indices_repeating(num_heads: int, head_dim: int, pruning_indices: torch.Tensor):
    all_indices = []
    for head_num in range(num_heads):
        all_indices.append(
            pruning_indices+head_num*head_dim)
    return torch.cat(all_indices)

In [ ]:
#q - attn heads
#k,v - kv heads

In [ ]:
def close_rope_pairs(local_idxs, head_dim):
    local_idxs = torch.as_tensor(local_idxs, dtype=torch.long)
    half = head_dim // 2
    paired = torch.where(
        local_idxs < half,
        local_idxs + half,
        local_idxs - half,
    )
    return torch.unique(torch.cat([local_idxs, paired]), sorted=True)

In [ ]:
idxs=[2, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
idxs = [2, 6] #define indexes per head_dim (due to the 4D tensors inside!)
idxs = close_rope_pairs(idxs, bridge.model.config.head_dim)  # for symmetric ROPE pruning in TP
repeated_idxs = manually_indices_repeating(
    bridge.model.config.num_attention_heads,
    bridge.model.config.head_dim,
    torch.tensor(idxs)
)

/tmp/ipykernel_1054387/2271329800.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(idxs)


In [ ]:
idxs

tensor([ 2,  6, 66, 70])

In [ ]:
repeated_idxs

tensor([   2,    6,   66,   70,  130,  134,  194,  198,  258,  262,  322,  326,
         386,  390,  450,  454,  514,  518,  578,  582,  642,  646,  706,  710,
         770,  774,  834,  838,  898,  902,  962,  966, 1026, 1030, 1090, 1094,
        1154, 1158, 1218, 1222, 1282, 1286, 1346, 1350, 1410, 1414, 1474, 1478,
        1538, 1542, 1602, 1606, 1666, 1670, 1730, 1734, 1794, 1798, 1858, 1862,
        1922, 1926, 1986, 1990, 2050, 2054, 2114, 2118, 2178, 2182, 2242, 2246,
        2306, 2310, 2370, 2374, 2434, 2438, 2498, 2502, 2562, 2566, 2626, 2630,
        2690, 2694, 2754, 2758, 2818, 2822, 2882, 2886, 2946, 2950, 3010, 3014,
        3074, 3078, 3138, 3142, 3202, 3206, 3266, 3270, 3330, 3334, 3394, 3398,
        3458, 3462, 3522, 3526, 3586, 3590, 3650, 3654, 3714, 3718, 3778, 3782,
        3842, 3846, 3906, 3910, 3970, 3974, 4034, 4038])

In [ ]:
bridge.blocks[0].attn.q._original_component.out_features

4096

In [ ]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs.tolist() )

In [ ]:
len(repeated_idxs)

128

In [ ]:
group.__len__()

59

In [ ]:
print(group) #todo rotate half duplicate indices?


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=128
[1] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _Reshape_1554(), len(idxs)=128
[2] prune_out_channels on _Reshape_1554() => prune_out_channels on _ElementWiseOp_1553(TransposeBackward0), len(idxs)=128
[3] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prune_out_channels on _Slice_1552(), len(idxs)=128
[4] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => 

In [ ]:
bridge.config.num_attention_heads
bridge.config.num_key_value_heads

8

In [ ]:
bridge.blocks[0].attn.config.n_key_value_heads #universal names for all models
bridge.blocks[0].attn.config.n_heads

32

In [ ]:
import torch_pruning
def map_q_indices_to_kv(
    q_idxs,
    *,
    num_q_heads,
    num_kv_heads,
    head_dim,
):
    assert num_q_heads % num_kv_heads == 0

    repeat = num_q_heads // num_kv_heads

    mapper = torch_pruning.dependency.index_mapping._GQAIndexMapping(
        repeat=repeat,
        head_dim=head_dim,
        reverse=True,
    )

    hybrid_q_idxs = [
        torch_pruning._helpers._HybridIndex(
            idx=int(idx),
            root_idx=int(idx),
        )
        for idx in q_idxs
    ]

    mapped_hybrid_idxs = mapper(hybrid_q_idxs)

    # GQA mapping является many-to-one:
    # несколько Q-heads отображаются на одну KV-head.
    kv_idxs = sorted({
        int(mapped.idx)
        for mapped in mapped_hybrid_idxs
    })

    return kv_idxs

In [ ]:
config = bridge.model.config

num_q_heads = config.num_attention_heads
num_kv_heads = config.num_key_value_heads
head_dim = config.head_dim

q_idxs = list(map(int, group[0][1]))

kv_idxs = map_q_indices_to_kv(
    q_idxs,
    num_q_heads=num_q_heads,
    num_kv_heads=num_kv_heads,
    head_dim=head_dim,
)
kv_idxs = torch.tensor(kv_idxs)

print("Q:", len(q_idxs), len(set(q_idxs)))
print("KV:", len(kv_idxs), len(set(kv_idxs)))
print("KV min/max:", min(kv_idxs), max(kv_idxs))

Q: 128 128
KV: 32 32
KV min/max: tensor(2) tensor(966)


In [ ]:
import torch_pruning


def replace_linear_indices(
    group,
    module,
    new_idxs,
    *,
    prune_out,
):
    new_idxs = sorted(set(map(int, new_idxs)))
    matched_items = []

    for i, (dep, old_idxs) in enumerate(group):
        if dep.target.module is not module:
            continue

        if prune_out:
            correct_handler = DG.is_out_channel_pruning_fn(
                dep.handler
            )
        else:
            correct_handler = DG.is_in_channel_pruning_fn(
                dep.handler
            )

        if not correct_handler:
            continue

        print(
            f"[{i}] {dep.target.name}: "
            f"{len(old_idxs)} → {len(new_idxs)}"
        )

        group[i] = torch_pruning._helpers.GroupItem(
            dep=dep,
            idxs=new_idxs,
        )

        matched_items.append(i)

    assert len(matched_items) == 1, (
        f"Expected one group item for {module}, "
        f"found {matched_items}"
    )

In [ ]:
bridge.blocks[0].attn.k

LinearBridge(4096 -> 992, bias=False, original_component=Linear)

In [ ]:
q_proj = bridge.blocks[0].attn.q._original_component
k_proj = bridge.blocks[0].attn.k._original_component
v_proj = bridge.blocks[0].attn.v._original_component
o_proj = bridge.blocks[0].attn.o._original_component

replace_linear_indices(
    group,
    k_proj,
    kv_idxs,
    prune_out=True,
)

replace_linear_indices(
    group,
    v_proj,
    kv_idxs,
    prune_out=True,
)

[56] model.layers.0._original_component.self_attn._original_component.k_proj._original_component (Linear(in_features=4096, out_features=1024, bias=False)): 128 → 32
[39] model.layers.0._original_component.self_attn._original_component.v_proj._original_component (Linear(in_features=4096, out_features=1024, bias=False)): 128 → 32


In [ ]:
# for dep, idx in group.items:
#     print(dep, idx)

In [ ]:
# 3. Do the pruning
if DG.check_pruning_group(group): # avoid over-pruning, i.e., channels=0.
    group.prune()
# 4. Save & Load
# model.zero_grad() # clear gradients to avoid a large file size
# torch.save(model, 'model.pth') # !! no .state_dict here since the structure has been changed after pruning
# model = torch.load('model.pth') # load the pruned model. you may need torch.load('model.pth', weights_only=False) for PyTorch 2.6.0+.


In [ ]:
len(repeated_idxs) / bridge.model.config.num_attention_heads

4.0

In [ ]:
bridge.model.config.head_dim = bridge.model.config.head_dim - (len(repeated_idxs) // bridge.model.config.num_attention_heads) 

In [ ]:
bridge.model.config.head_dim

124

In [ ]:
bridge.blocks[0].attn._original_component.head_dim

128

Manual update of static params

In [ ]:
bridge.blocks[0].attn._original_component.head_dim = bridge.model.config.head_dim

Manual rope resize

In [ ]:
bridge.blocks[0].attn.hook_cos.remove_hooks()

NOTE: call it for PruningInstrumentor too! (symmetric pairs zeroed in cos/sin)

In [ ]:
def make_activation_prune_hook(pruned_idxs, original_size):
    keep_idxs = [
        idx
        for idx in range(original_size)
        if idx not in pruned_idxs
    ]

    def hook(activation, hook): #cos or sin
        index = torch.tensor(
            keep_idxs,
            device=activation.device,
            dtype=torch.long,
        )

        return activation.index_select(
            dim=-1,
            index=index,
        )

    return hook

def make_activation_mask_hook(pruned_idxs, original_size):
    def hook(cos_or_sin, hook):
        mask = torch.ones(
            original_size,
            device=cos_or_sin.device,
            dtype=cos_or_sin.dtype,
        )
        mask[pruned_idxs.to(cos_or_sin.device)] = 0
        return cos_or_sin * mask

    return hook


def rotate_half_indices(pruned_idxs, head_dim: int) -> torch.Tensor:
    if head_dim % 2 != 0:
        raise ValueError(f"head_dim must be even, got {head_dim}")

    idxs = torch.as_tensor(pruned_idxs, dtype=torch.long)
    half = head_dim // 2

    if torch.any((idxs < 0) | (idxs >= head_dim)):
        raise IndexError("pruned_idxs contains indices outside head_dim")

    return torch.where(idxs < half, idxs + half, idxs - half)




def prepare_rope(idxs, head_dim_original, attn_bridge, prune_hook_fn=make_activation_prune_hook):
    # pruned_idxs = close_rope_pairs(idxs, head_dim) #DONT USE IF ALREADY ROPED
    #also cant use symmetric indices in cos/sin and non symmetric
    #indices in q/k vectors at the same time.
    pruned_idxs = idxs
    mask = torch.ones(head_dim_original)
    mask[pruned_idxs] = 0

    attn = attn_bridge
    
    attn.hook_cos.add_hook(
         prune_hook_fn(pruned_idxs, head_dim_original)
    )
        
    attn.hook_sin.add_hook(
        prune_hook_fn(pruned_idxs, head_dim_original)
        #can make without rotate if indexes already symmetric pairs
        #make_activation_prune_hook(rotate_half_indices(pruned_idxs, head_dim), head_dim)
    )

In [ ]:
assert idxs.__len__() == 4

In [ ]:
prepare_rope(idxs, head_dim, bridge.blocks[0].attn, prune_hook_fn=make_activation_prune_hook)

In [ ]:
# bridge.reset_hooks()

In [ ]:
# bridge.blocks[0].attn.hook_cos.remove_hooks()

In [ ]:
bridge.blocks[0].attn.hook_cos.fwd_hooks.__len__()

1

print pruning functions & layers

In [ ]:
bridge.model.layers[0]._original_component.self_attn._original_component.q_proj

LinearBridge(4096 -> 3968, bias=False, original_component=Linear)

In [ ]:
bridge.get_submodule("blocks.0.attn").config.head_dim

128

In [ ]:
m_e = bridge.get_submodule("blocks.0.attn.k")   

In [ ]:
for (name, module) in bridge.named_modules():
    # print(name, )
    # if (type(module) != TransformerBridge):
    #     print(str(module))
    if (module == m_e):
        print(name, module)
        print(module.hook_in.name) #костыль через хук
    

blocks.0._original_component.self_attn._original_component.k_proj LinearBridge(4096 -> 992, bias=False, original_component=Linear)
blocks.0.attn.k.hook_in


In [ ]:
count = 0
for i, (dep, idxs_t) in enumerate(group):
    layer = dep.layer
    pruning_fn = dep.pruning_fn
    print(layer, pruning_fn, len(idxs_t))
    if (not type(dep.layer).__module__.startswith("torch_pruning.ops")):
        if ((not dep.pruning_fn.__name__.endswith("in_channels")) or (i == 0)):
            print("Add to our hooks! ", count)
            print(dep.target.module, type(dep.target.module))
            count += 1
        print(dep.target.name)
        print

Linear(in_features=4096, out_features=3968, bias=False) <bound method LinearPruner.prune_out_channels of <torch_pruning.pruner.function.LinearPruner object at 0x7d33359a1510>> 128
Add to our hooks!  0
Linear(in_features=4096, out_features=3968, bias=False) <class 'torch.nn.modules.linear.Linear'>
model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=3968, bias=False))
_Reshape_1554() <bound method DummyPruner.prune_out_channels of <torch_pruning.ops.ReshapePruner object at 0x7d333598edd0>> 128
_ElementWiseOp_1553(TransposeBackward0) <bound method DummyPruner.prune_out_channels of <torch_pruning.ops.ElementWisePruner object at 0x7d333598cb20>> 128
_Slice_1552() <bound method SlicePruner.prune_out_channels of <torch_pruning.ops.SlicePruner object at 0x7d333598fb50>> 128
_Slice_1558() <bound method SlicePruner.prune_out_channels of <torch_pruning.ops.SlicePruner object at 0x7d333598fb50>> 128
_ElementWiseOp_1548(

In [ ]:
torch_pruning.ops._ReshapeOp

torch_pruning.ops._ReshapeOp

In [ ]:
from utils import evaluate_language_model
bridge.reset_hooks()
baseline_metrics = evaluate_language_model(
    bridge,
    evaluation_blocks,
    batch_size=EVAL_BATCH_SIZE,
)
baseline_metrics

{'loss': 2.273681640625, 'perplexity': 9.715102195739746}

In [ ]:
bridge.blocks[0].attn.o._original_component

Linear(in_features=3968, out_features=4096, bias=False)

In [ ]:
bridge.blocks[0].attn.o._original_component.weight.shape

torch.Size([4096, 3968])

TODO: there is possibility to prune (k and q) and (v and o) independently. Now due to the bug of Torch Pruning they prune as one group with shared idxs.

But it good, because they all share the same head_dim parameter. It would be hard to create different head_dim for different k and o prunings.

In [ ]:
print(bridge.blocks[0].attn.v._original_component)
print(bridge.blocks[0].attn.k._original_component)
print(bridge.blocks[0].attn.q._original_component)
print(bridge.blocks[0].attn.o._original_component)

Linear(in_features=4096, out_features=992, bias=False)
Linear(in_features=4096, out_features=992, bias=False)
Linear(in_features=4096, out_features=3968, bias=False)
Linear(in_features=3968, out_features=4096, bias=False)


For now we will consider them as one group (it works fine also for TransferPruning, where activation change also affect on all 4 (k, v, o, q) parameters).

In [ ]:
#metrics of original model
pruning_zero_metrics = {
    'loss': 2.26513671875,
    'perplexity': 9.632441520690918,
}

for idxs=[2, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

{'loss': 6.7177734375, 'perplexity': 826.97412109375}

for idx = [2, 6]  (when prune cos and sin in rope manually, not in-fly)  
{'loss': 3.18994140625, 'perplexity': 24.287004470825195}

huge increase because it wrong goes to partial rope implementation when rope.size < head_dim, it's totally wrong application of freqs and cos to part of embedding.

for idx = [2, 6] (masked rotary cos and sin for only concrete layer)  
{'loss': 2.308349609375, 'perplexity': 10.057811737060547}


Seems like correct little increase!

for idx [2, 6] with rotate sin in rope

{'loss': 2.328857421875, 'perplexity': 10.266204833984375}


More correct with symmetric prune q/k indices (and rope too):  




{'loss': 2.273681640625, 'perplexity': 9.715102195739746}

In [ ]:
group

In [ ]:
model2 = load_model(device="cuda:3")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
bridge2 = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model2,
    dtype=torch.float16,
)

## Compare with transfer pruning of attn.q, k, v, o

The same channels should be pruned

In [ ]:
bridge_for_prun_ins = bridge2

In [ ]:
modules_to_prune = [
    bridge_for_prun_ins.blocks[0].attn.q,
    bridge_for_prun_ins.blocks[0].attn.k,
    bridge_for_prun_ins.blocks[0].attn.v,
    bridge_for_prun_ins.blocks[0].attn.o,
]
#TODO manual passing repeated indexes??? make interface that get indexes in terms of one head_dim and automatically repeat them??

#there is no bias in Llama, we can compare it
#also there is no need to hook for cos, because prune activations, not weights/cos
#q_out
rescale_param = False
PruningInstrumentor.prepare_linear_for_pruning(bridge_for_prun_ins.blocks[0].attn.q, torch.tensor([]).to(bridge_for_prun_ins.device), repeated_idxs.to(bridge_for_prun_ins.device), rescale=rescale_param)
#k_out - auto? Maybe... but no, due to the apply_rotary_pos_emb k*cos + rot_half(k)*sin
PruningInstrumentor.prepare_linear_for_pruning(bridge_for_prun_ins.blocks[0].attn.k, torch.tensor([]).to(bridge_for_prun_ins.device), kv_idxs.to(bridge_for_prun_ins.device), rescale=rescale_param)
#v_out - need to prune because TP also prunes it
PruningInstrumentor.prepare_linear_for_pruning(bridge_for_prun_ins.blocks[0].attn.v, torch.tensor([]).to(bridge_for_prun_ins.device), kv_idxs.to(bridge_for_prun_ins.device), rescale=rescale_param)
#o_in - dont need, because already prune v_out (checked!!!)
#PruningInstrumentor.prepare_linear_for_pruning(bridge_for_prun_ins.blocks[0].attn.o, kv_idxs.to(bridge_for_prun_ins.device), torch.tensor([]).to(bridge_for_prun_ins.device))


In [ ]:
head_dim

128

In [ ]:
assert idxs.__len__() == 4

In [ ]:
if (False): #dont need to do it in case of Activation Pruning
    prepare_rope(idxs, head_dim, bridge_for_prun_ins.blocks[0].attn, prune_hook_fn=make_activation_mask_hook)

In [ ]:
bridge_for_prun_ins.blocks[0].attn.q.hook_out.fwd_hooks

[LensHandle(hook=<torch.utils.hooks.RemovableHandle object at 0x7d375423a020>, is_permanent=False, context_level=None, user_hook=<function PruningInstrumentor._flatten_heads_wrapper.<locals>.wrappedHook at 0x7d37570cadd0>)]

In [ ]:
#bridge_for_prun_ins.reset_hooks()

In [ ]:
from utils import evaluate_language_model
our_metrics = evaluate_language_model(
    bridge_for_prun_ins,
    evaluation_blocks.to(bridge_for_prun_ins.device),
    batch_size=EVAL_BATCH_SIZE,
)
our_metrics

{'loss': 2.273681640625, 'perplexity': 9.715102195739746}

PruningInstumentor with rescaling!


{'loss': 2.272705078125, 'perplexity': 9.705619812011719} for transfer pruning with q, k, v pruned in blocks.0.attn

{'loss': 2.27099609375, 'perplexity': 9.689046859741211}  
for transfer pruning with only q and v (k and rotate_half doesnt prune)

{'loss': 2.273193359375, 'perplexity': 9.710359573364258}
for qkv and without norm scaling

In [ ]:
print("baseline_metrics:", baseline_metrics)
print("our_metrics:", our_metrics)

baseline_metrics: {'loss': 2.273681640625, 'perplexity': 9.715102195739746}
our_metrics: {'loss': 2.273681640625, 'perplexity': 9.715102195739746}


baseline_metrics: {'loss': 2.273681640625, 'perplexity': 9.715102195739746}
our_metrics: {'loss': 2.273681640625, 'perplexity': 9.715102195739746}

# CONGRATULATIONS, THEY ARE EQUAL!!!

In [ ]:
# Compare bridge and bridge_for_prun_ins at every attention HookPoint.

In [ ]:
# Compare bridge and bridge_for_prun_ins at every attention HookPoint.
from collections import OrderedDict, defaultdict
from transformer_lens.hook_points import HookPoint

def capture_attention_stages(model, layer_idx, forward_fn):
    captured, counts, handles = OrderedDict(), defaultdict(int), []
    for relative_name, module in model.blocks[layer_idx].attn.named_modules():
        if not isinstance(module, HookPoint):
            continue
        def capture(value, hook, base_name=relative_name):
            if isinstance(value, torch.Tensor):
                call_idx = counts[base_name]
                counts[base_name] += 1
                name = base_name if call_idx == 0 else f"{base_name}#{call_idx}"
                captured[name] = value.detach().float().cpu().clone()
            return value
        # Added after pruning hooks, so this sees the value consumed downstream.
        module.add_hook(capture, dir="fwd")
        #handles.append(module.add_hook(capture, dir="fwd"))
    try:
        output = forward_fn(model)
        if isinstance(output, torch.Tensor):
            captured["<model_output>"] = output.detach().float().cpu().clone()
    finally:
        # for handle in handles:
        #     handle.remove()  # preserve all pre-existing pruning hooks
        model.reset_hooks()
    return captured

def _align_larger_to_smaller(smaller, larger, index_candidates):
    if smaller.ndim != larger.ndim:
        return None, None
    dims = [d for d, (a, b) in enumerate(zip(smaller.shape, larger.shape)) if a != b]
    if len(dims) != 1:
        return None, None
    dim = dims[0]
    if smaller.shape[dim] >= larger.shape[dim]:
        return None, None
    removed = larger.shape[dim] - smaller.shape[dim]
    for label, raw_idxs in index_candidates.items():
        pruned = torch.as_tensor(raw_idxs, dtype=torch.long).unique(sorted=True)
        pruned = pruned[(pruned >= 0) & (pruned < larger.shape[dim])]
        if len(pruned) != removed:
            continue
        keep = torch.ones(larger.shape[dim], dtype=torch.bool)
        keep[pruned] = False
        return larger.index_select(dim, keep.nonzero(as_tuple=False).flatten()), f"{label}, dim={dim}"
    return None, None

def compare_attention_stages(reference, candidate, index_candidates=None, atol=1e-4, rtol=1e-3):
    index_candidates, first, rows = index_candidates or {}, None, []
    names = list(reference) + [name for name in candidate if name not in reference]
    for stage_idx, name in enumerate(names):
        if name not in reference or name not in candidate:
            print(f"[{stage_idx:02d}] {name:32s} MISSING ref={name in reference} test={name in candidate}")
            first = first or name
            continue
        ref, test, alignment = reference[name], candidate[name], None
        original_shapes = tuple(ref.shape), tuple(test.shape)

        # LinearBridge hook conversion may expose one model as [B, S, H, D]
        # while a structurally-pruned Linear is captured as [B, S, H*D].
        # Canonicalize only this unambiguous one-extra-head-axis case.
        if ref.ndim + 1 == test.ndim and ref.shape[:-1] == test.shape[:-2]:
            test = test.flatten(-2)
            alignment = "flatten candidate [H,D]->[H*D]"
        elif test.ndim + 1 == ref.ndim and test.shape[:-1] == ref.shape[:-2]:
            ref = ref.flatten(-2)
            alignment = "flatten reference [H,D]->[H*D]"

        if ref.shape != test.shape:
            previous_alignment = alignment
            aligned, index_alignment = _align_larger_to_smaller(ref, test, index_candidates)
            if aligned is not None:
                test = aligned
                alignment = " + ".join(filter(None, [previous_alignment, index_alignment]))
            else:
                aligned, index_alignment = _align_larger_to_smaller(test, ref, index_candidates)
                if aligned is not None:
                    ref = aligned
                    alignment = " + ".join(filter(None, [previous_alignment, index_alignment]))
        if ref.shape != test.shape:
            print(f"[{stage_idx:02d}] {name:32s} SHAPE ref={original_shapes[0]} test={original_shapes[1]}")
            first = first or name
            continue
        delta = test - ref
        close = torch.allclose(ref, test, atol=atol, rtol=rtol)
        max_abs = delta.abs().max().item() if delta.numel() else 0.0
        mean_abs = delta.abs().mean().item() if delta.numel() else 0.0
        rmse = delta.square().mean().sqrt().item() if delta.numel() else 0.0
        rel_l2 = delta.norm().item() / max(ref.norm().item(), 1e-12)
        abs_l2 = delta.square().sum().sqrt().item()
        status = "OK" if close else "DIFF"
        suffix = f" aligned={alignment}" if alignment else ""
        print(f"[{stage_idx:02d}] {name:32s} {status:4s} shape={tuple(ref.shape)} max={max_abs:.3e} mean={mean_abs:.3e} rmse={rmse:.3e} rel_l2={rel_l2:.3e} abs_l2={abs_l2:.3e}{suffix}")
        rows.append({"stage": name, "close": close, "reference_shape": original_shapes[0], "candidate_shape": original_shapes[1], "alignment": alignment, "max_abs": max_abs, "mean_abs": mean_abs, "rmse": rmse, "rel_l2": rel_l2, "abs_l2": abs_l2})
        if not close and first is None:
            first = name
    print("\nFIRST DIVERGENCE:", first)
    return first, rows


In [ ]:
comparison_layer = 0
comparison_tokens = evaluation_blocks[0][:1].to(DEVICE)
bridge.eval()
bridge_for_prun_ins.eval()
forward_fn = lambda model: model(comparison_tokens, return_type="logits")

tp_stages = capture_attention_stages(bridge, comparison_layer, forward_fn)
comparison_tokens = evaluation_blocks[0][:1].to(bridge_for_prun_ins.device)
instrumentor_stages = capture_attention_stages(bridge_for_prun_ins, comparison_layer, forward_fn)

# Selects the index set whose length matches the structural-vs-masked shape delta.
alignment_candidates = {
    "local_head_idxs": torch.as_tensor(idxs),
    "q_flat_idxs": torch.as_tensor(repeated_idxs),
    "kv_flat_idxs": torch.as_tensor(kv_idxs),
}
first_divergence, comparison_rows = compare_attention_stages(
    tp_stages,
    instrumentor_stages,
    alignment_candidates,
    # atol=2e-5,
    # rtol=2e-4,
)


[00] hook_in                          OK   shape=(1, 1, 4096) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
[01] _original_component.q_proj.hook_in OK   shape=(1, 1, 4096) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
[02] _original_component.q_proj.hook_out OK   shape=(1, 1, 3968) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00 aligned=flatten candidate [H,D]->[H*D] + q_flat_idxs, dim=2
[03] _original_component.k_proj.hook_in OK   shape=(1, 1, 4096) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
[04] _original_component.k_proj.hook_out OK   shape=(1, 1, 992) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00 aligned=flatten candidate [H,D]->[H*D] + q_flat_idxs, dim=2
[05] _original_component.v_proj.hook_in OK   shape=(1, 1, 4096) max=0.000e+00 mean=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
[06] _original_component.v_pr

In [ ]:
delta = instrumentor_stages['_original_component.o_proj.hook_out'] - tp_stages['_original_component.o_proj.hook_out']
print("we see that in report", delta.abs().mean().item())

we see that in report 2.9831426218152046e-08


#done, causes of that delta inspected below
solution: increase tolerance (atol) to 1e-4 and rtol to 1e-3

## !!! But at the end model output have huge difference. (at least in absolute measure)

rel_l2=3.985e-03 abs_l2=5.609e+00

In [ ]:
instrumentor_stages['<model_output>']

tensor([[[-22.0000, -16.8750, -15.7500,  ...,  10.4375,  10.4375,  10.4375]]])

In [ ]:
import torch
import torch.nn.functional as F


def print_tensor_delta(name, reference, candidate):
    assert reference.shape == candidate.shape, (
        name,
        reference.shape,
        candidate.shape,
    )

    reference_f32 = reference.detach().float()
    candidate_f32 = candidate.detach().float()
    delta = candidate_f32 - reference_f32

    max_abs = delta.abs().max().item()
    mean_abs = delta.abs().mean().item()
    rmse = delta.square().mean().sqrt().item()
    rel_l2 = delta.norm().item() / max(
        reference_f32.norm().item(),
        1e-12,
    )

    print(
        f"{name}\n"
        f"  shape:   {tuple(reference.shape)}\n"
        f"  max:     {max_abs:.8e}\n"
        f"  mean:    {mean_abs:.8e}\n"
        f"  rmse:    {rmse:.8e}\n"
        f"  rel_l2:  {rel_l2:.8e}\n"
    )


layer_idx = 0

tp_o = bridge.blocks[layer_idx].attn.o._original_component
ins_o = bridge_for_prun_ins.blocks[layer_idx].attn.o._original_component

print("TP o_proj:", tp_o)
print("Instrumentor o_proj:", ins_o)
print("TP weight:", tuple(tp_o.weight.shape))
print("Instrumentor weight:", tuple(ins_o.weight.shape))

TP o_proj: Linear(in_features=3968, out_features=4096, bias=False)
Instrumentor o_proj: Linear(in_features=4096, out_features=4096, bias=False)
TP weight: (4096, 3968)
Instrumentor weight: (4096, 4096)


In [ ]:
# Captured values находятся на CPU в float32.
tp_o_input_cpu = tp_stages[
    "_original_component.o_proj.hook_in"
]

ins_o_input_cpu = instrumentor_stages[
    "_original_component.o_proj.hook_in"
]

# TransformerLens hook conversion может показать O input как [B, S, H, D].
if ins_o_input_cpu.ndim == tp_o_input_cpu.ndim + 1:
    ins_o_input_cpu = ins_o_input_cpu.flatten(-2)

print("TP input:", tuple(tp_o_input_cpu.shape))
print("Instrumentor input:", tuple(ins_o_input_cpu.shape))

assert tp_o_input_cpu.shape[-1] == tp_o.in_features
assert ins_o_input_cpu.shape[-1] == ins_o.in_features

# Для o_proj после GQA используются индексы Q-head пространства.
pruned_o_input_idxs = torch.as_tensor(
    repeated_idxs,
    dtype=torch.long,
).cpu().unique(sorted=True)

keep_mask = torch.ones(
    ins_o.in_features,
    dtype=torch.bool,
)
keep_mask[pruned_o_input_idxs] = False
keep_o_input_idxs = keep_mask.nonzero(
    as_tuple=False,
).flatten()

print("Removed:", len(pruned_o_input_idxs))
print("Kept:", len(keep_o_input_idxs))

assert len(keep_o_input_idxs) == tp_o.in_features

TP input: (1, 1, 3968)
Instrumentor input: (1, 1, 4096)
Removed: 128
Kept: 3968


In [ ]:
ins_o_input_compact_cpu = ins_o_input_cpu.index_select(
    -1,
    keep_o_input_idxs,
)

ins_o_weight_compact_cpu = (
    ins_o.weight.detach().cpu().index_select(
        1,
        keep_o_input_idxs,
    )
)

print_tensor_delta(
    "O input: TP compact vs Instrumentor keep",
    tp_o_input_cpu,
    ins_o_input_compact_cpu,
)

print_tensor_delta(
    "O weight: TP compact vs Instrumentor keep",
    tp_o.weight.detach().cpu(),
    ins_o_weight_compact_cpu,
)

print(
    "Removed Instrumentor input max:",
    ins_o_input_cpu[..., pruned_o_input_idxs]
    .abs()
    .max()
    .item(),
)

print(
    "Removed Instrumentor input norm:",
    ins_o_input_cpu[..., pruned_o_input_idxs]
    .norm()
    .item(),
)

O input: TP compact vs Instrumentor keep
  shape:   (1, 1, 3968)
  max:     0.00000000e+00
  mean:    0.00000000e+00
  rmse:    0.00000000e+00
  rel_l2:  0.00000000e+00

O weight: TP compact vs Instrumentor keep
  shape:   (4096, 3968)
  max:     0.00000000e+00
  mean:    0.00000000e+00
  rmse:    0.00000000e+00
  rel_l2:  0.00000000e+00

Removed Instrumentor input max: 0.0
Removed Instrumentor input norm: 0.0


In [ ]:
torch.cuda.empty_cache()

NameError: name 'gc' is not defined

In [ ]:
tp_o.weight.shape

torch.Size([4096, 3968])

In [ ]:
ins_o.weight.shape

torch.Size([4096, 4096])

In [ ]:
import gc
device = tp_o.weight.device
dtype = tp_o.weight.dtype

tp_o_input = tp_o_input_cpu.to(
    device=device,
    dtype=dtype,
)

ins_o_input_full = ins_o_input_cpu.to(
    device="cuda:3",
    dtype=dtype,
)

keep_device = keep_o_input_idxs.to("cuda:3")


# Для надёжности явно зануляем нужные координаты.
ins_o_input_zeroed = ins_o_input_full.clone()
ins_o_input_zeroed[..., pruned_o_input_idxs] = 0

with torch.inference_mode():
    # Реальный structural o_proj: K=3968.
    output_tp = tp_o(tp_o_input).to("cpu")

    # Реальный полный o_proj: K=4096, часть входов равна нулю.
    output_ins_zeroed = ins_o(ins_o_input_zeroed).to("cpu")

    # Полный вес Instrumentor вручную сжат до K=3968.
    output_ins_compact = F.linear(
        ins_o_input_zeroed.index_select(-1, keep_device),
        ins_o.weight.index_select(1, keep_device),
        ins_o.bias,
    ).to("cpu")

print_tensor_delta(
    f"Native {dtype}: TP structural vs Instrumentor zeroed",
    output_tp.to("cpu"),
    output_ins_zeroed.to("cpu"),
)

print_tensor_delta(
    f"Native {dtype}: TP structural vs Instrumentor compact",
    output_tp,
    output_ins_compact,
)

print_tensor_delta(
    f"Native {dtype}: Instrumentor compact vs zeroed",
    output_ins_compact,
    output_ins_zeroed,
)

Native torch.bfloat16: TP structural vs Instrumentor zeroed
  shape:   (1, 1, 4096)
  max:     3.05175781e-05
  mean:    2.98314262e-08
  rmse:    8.92082596e-07
  rel_l2:  1.10665289e-04

Native torch.bfloat16: TP structural vs Instrumentor compact
  shape:   (1, 1, 4096)
  max:     0.00000000e+00
  mean:    0.00000000e+00
  rmse:    0.00000000e+00
  rel_l2:  0.00000000e+00

Native torch.bfloat16: Instrumentor compact vs zeroed
  shape:   (1, 1, 4096)
  max:     3.05175781e-05
  mean:    2.98314262e-08
  rmse:    8.92082596e-07
  rel_l2:  1.10665289e-04



DONE: why DIFF? (Especially between Instrumentor compact and zeroed versions). Different kernels for diff shapes? tp32 tp16 conversions? maybe in hooks bridges conversions?

Answer: different zeros positions leads to mistakes during partial sums inside kernels (most probably...).

In [ ]:
x_full = ins_o_input_zeroed.contiguous()
w_full = ins_o.weight.contiguous()

x_compact = x_full.index_select(
    -1,
    keep_device,
).contiguous()

w_compact = w_full.index_select(
    1,
    keep_device,
).contiguous()

with torch.inference_mode():
    out_full_f = F.linear(
        x_full,
        w_full,
        ins_o.bias,
    )

    out_compact_f = F.linear(
        x_compact,
        w_compact,
        ins_o.bias,
    )

print_tensor_delta(
    "F.linear compact vs F.linear full-zeroed",
    out_compact_f.cpu(),
    out_full_f.cpu(),
)

F.linear compact vs F.linear full-zeroed
  shape:   (1, 1, 4096)
  max:     3.05175781e-05
  mean:    2.98314262e-08
  rmse:    8.92082596e-07
  rel_l2:  1.10665289e-04



In [ ]:
padding = x_full.shape[-1] - x_compact.shape[-1]
assert padding == len(pruned_o_input_idxs)

x_trailing_zeros = F.pad(
    x_compact,
    (0, padding),
).contiguous()

w_trailing_zeros = F.pad(
    w_compact,
    (0, padding),
).contiguous()

with torch.inference_mode():
    out_compact = F.linear(
        x_compact,
        w_compact,
        ins_o.bias,
    )

    out_interspersed_zeros = F.linear(
        x_full,
        w_full,
        ins_o.bias,
    )

    out_trailing_zeros = F.linear(
        x_trailing_zeros,
        w_trailing_zeros,
        ins_o.bias,
    )

print_tensor_delta(
    "compact K=3968 vs trailing zeros K=4096",
    out_compact.cpu(),
    out_trailing_zeros.cpu(),
)

print_tensor_delta(
    "interspersed vs trailing zeros, both K=4096",
    out_interspersed_zeros.cpu(),
    out_trailing_zeros.cpu(),
)

compact K=3968 vs trailing zeros K=4096
  shape:   (1, 1, 4096)
  max:     0.00000000e+00
  mean:    0.00000000e+00
  rmse:    0.00000000e+00
  rel_l2:  0.00000000e+00

interspersed vs trailing zeros, both K=4096
  shape:   (1, 1, 4096)
  max:     3.05175781e-05
  mean:    2.98314262e-08
  rmse:    8.92082596e-07
  rel_l2:  1.10665314e-04



interspersed != trailing  
It means if zeroes go together or mixed inside tensor, there is some diff between calculations or kernels.

In [ ]:
import torch
import torch.nn.functional as F

from torch.profiler import (
    profile,
    ProfilerActivity,
    record_function,
)

# Предварительный прогрев — исключаем инициализацию cuBLAS и compilation.
for _ in range(10):
    F.linear(x_full, w_full, ins_o.bias)
    F.linear(x_trailing_zeros, w_trailing_zeros, ins_o.bias)
    F.linear(x_compact, w_compact, ins_o.bias)

torch.cuda.synchronize()

with profile(
    activities=[
        ProfilerActivity.CPU,
        ProfilerActivity.CUDA,
    ],
    record_shapes=True,
    with_stack=True,
) as prof:
    with torch.inference_mode():
        with record_function("GEMM_INTERSPERSED_K4096"):
            out_interspersed = F.linear(
                x_full,
                w_full,
                ins_o.bias,
            )
            torch.cuda.synchronize()

        with record_function("GEMM_TRAILING_K4096"):
            out_trailing = F.linear(
                x_trailing_zeros,
                w_trailing_zeros,
                ins_o.bias,
            )
            torch.cuda.synchronize()

        with record_function("GEMM_COMPACT_K3968"):
            out_compact = F.linear(
                x_compact,
                w_compact,
                ins_o.bias,
            )
            torch.cuda.synchronize()

/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [ ]:
print(
    prof.key_averages(
        group_by_input_shape=True,
    ).table(
        sort_by="self_cuda_time_total",
        row_limit=50,
    )
)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------------------------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls                          Input Shapes  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------------------------------  
ampere_bf16_s16816gemm_bf16_64x64_sliced1x2_ldg8_f2f...         0.00%       0.000us         0.00%       0.000us       0.000us     116.864us       100.00%     116.864us      38.955us             3                                    []  
                                               aten::mm

In [ ]:
trace_path = (
    "/glazkov-dev/LoRa-Transfer-Pruning/"
    "experiments/o_proj_gemm_trace.json"
)
if (False):
    prof.export_chrome_trace(trace_path)

print(trace_path)

/glazkov-dev/LoRa-Transfer-Pruning/experiments/o_proj_gemm_trace.json


In [ ]:
if (False):
    torch.save(
        {
            "x_interspersed": x_full.detach().cpu(),
            "w_interspersed": w_full.detach().cpu(),
            "x_trailing": x_trailing_zeros.detach().cpu(),
            "w_trailing": w_trailing_zeros.detach().cpu(),
            "x_compact": x_compact.detach().cpu(),
            "w_compact": w_compact.detach().cpu(),
            "bias": (
                None
                if ins_o.bias is None
                else ins_o.bias.detach().cpu()
            ),
        },
        "/tmp/o_proj_gemm_inputs.pt",
    )

Try to find zero borders

In [ ]:
def insert_internal_gap_with_trailing_zeros(
    x_compact,
    w_compact,
    gap_start,
    internal_gap_size,
    full_k=4096,
):
    compact_k = x_compact.shape[-1]
    total_zeros = full_k - compact_k

    assert 0 <= internal_gap_size <= total_zeros
    assert 0 <= gap_start <= compact_k

    x_full = torch.zeros(
        *x_compact.shape[:-1],
        full_k,
        dtype=x_compact.dtype,
        device=x_compact.device,
    )

    w_full = torch.zeros(
        w_compact.shape[0],
        full_k,
        dtype=w_compact.dtype,
        device=w_compact.device,
    )

    # До внутреннего gap.
    x_full[..., :gap_start] = x_compact[..., :gap_start]
    w_full[:, :gap_start] = w_compact[:, :gap_start]

    # После внутреннего gap. Оставшиеся нули автоматически окажутся в хвосте.
    destination = gap_start + internal_gap_size

    x_full[
        ...,
        destination:destination + compact_k - gap_start,
    ] = x_compact[..., gap_start:]

    w_full[
        :,
        destination:destination + compact_k - gap_start,
    ] = w_compact[:, gap_start:]

    return x_full.contiguous(), w_full.contiguous()

In [ ]:
with torch.inference_mode():
    compact_reference = F.linear(
        x_compact,
        w_compact,
        ins_o.bias,
    )

gap_start = 1000

for internal_gap_size in [
    0,
    1,
    2,
    4,
    8,
    15,
    16,
    17,
    32,
    64,
    128,
]:
    x_test, w_test = insert_internal_gap_with_trailing_zeros(
        x_compact,
        w_compact,
        gap_start=gap_start,
        internal_gap_size=internal_gap_size,
    )

    with torch.inference_mode():
        output = F.linear(
            x_test,
            w_test,
            ins_o.bias,
        )

    delta = output.float() - compact_reference.float()

    print(
        f"gap={internal_gap_size:3d}",
        f"max={delta.abs().max().item():.8e}",
        f"different={(output != compact_reference).sum().item()}",
    )

gap=  0 max=0.00000000e+00 different=0
gap=  1 max=1.52587891e-05 different=2
gap=  2 max=3.05175781e-05 different=4
gap=  4 max=3.05175781e-05 different=6
gap=  8 max=1.52587891e-05 different=3
gap= 15 max=1.19209290e-07 different=2
gap= 16 max=3.05175781e-05 different=6
gap= 17 max=3.05175781e-05 different=4
gap= 32 max=3.05175781e-05 different=2
gap= 64 max=0.00000000e+00 different=0
gap=128 max=0.00000000e+00 different=0


## Important conclusion:
Position of zeroes in activation or weight and internal GEMM implementation, that split matrix operations on smaller with another shape, can lead to some errors in rounding.

((a + b) + (c + d)) can be != (a + (b + c) + d)

Accuracy depends on zeros/their positions/gemm operation.

But it not more than 10^-4

In [ ]:
#can affect to precision, but in this case, I checked, it doesn't affect.
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction,

(True,)

In [ ]:
bridge.blocks[0].attn.config.use_attn_result

как-то запрунить rope надо! (done)

In [ ]:
bridge.model.layers[0].self_attn.q._original_component

In [ ]:
bridge.model.layers[0].self_attn.q._original_component.weight.shape

In [ ]:
group_q = group[0][0].target.module
bridge_q = bridge.model.layers[0].self_attn.q._original_component

print("same module:", group_q is bridge_q)
print("group module id:", id(group_q))
print("bridge module id:", id(bridge_q))

print("group weight:", group_q.weight.shape)
print("bridge weight:", bridge_q.weight.shape)
print("same parameter:", group_q.weight is bridge_q.weight)

In [ ]:
q_linear = bridge.model.layers[0].self_attn.q._original_component

def inspect_q_output(module, inputs, output):
    print("ACTIVE Q MODULE:", id(module))
    print("ACTIVE WEIGHT:", tuple(module.weight.shape))
    print("Q OUTPUT:", tuple(output.shape))

handle = q_linear.register_forward_hook(inspect_q_output)

with torch.no_grad():
    _ = bridge.run_with_hooks(evaluation_blocks[:1].to(DEVICE))

handle.remove()

In [ ]:
import copy
import torch_pruning as tp

original = bridge.model.layers[0].self_attn.q._original_component
test_linear = copy.deepcopy(original)

print("before:", test_linear, test_linear.weight.shape)

tp.prune_linear_out_channels(
    test_linear,
    idxs=list(range(448)),
)

print("after:", test_linear, test_linear.weight.shape)

In [ ]:
q = bridge.model.layers[0].self_attn.q._original_component
for label, module in {
    "q linear": q,
    "q bridge": bridge.model.layers[0].self_attn.q,
    "attention bridge": bridge.model.layers[0].self_attn,
    "HF attention": bridge.model.layers[0].self_attn._original_component,
}.items():
    hook = getattr(module, "_hf_hook", None)
    print("\n", label, hook)

    if hook is not None:
        print("offload:", getattr(hook, "offload", None))
        print("execution_device:", getattr(hook, "execution_device", None))

        weights_map = getattr(hook, "weights_map", None)
        if weights_map is not None:
            print("keys:", list(weights_map.keys())[:20])

In [ ]:
print("no shapes error!")

In [ ]:
# Run on a freshly loaded model instead of group.prune(). This is destructive.
import torch.nn as nn

def linear_state(module):
    return (module.in_features, module.out_features, tuple(module.weight.shape), id(module.weight))

aliases = {
    "blocks": bridge.blocks[0].attn.q._original_component,
    "model.layers": bridge.model.layers[0].self_attn.q._original_component,
    "hf q_proj": bridge.model.layers[0].self_attn._original_component.q_proj._original_component,
    "group root": group[0][0].target.module,
}
for name, module in aliases.items():
    print(name, id(module), linear_state(module))
print("all modules identical:", len({id(m) for m in aliases.values()}) == 1)

def debug_group_prune_step_by_step(group):
    tracked = {id(dep.target.module): dep.target.module for dep, _ in group if isinstance(dep.target.module, nn.Linear)}
    previous = {key: linear_state(module) for key, module in tracked.items()}
    for step, (dep, idxs) in enumerate(group):
        dep(idxs)
        for key, module in tracked.items():
            current = linear_state(module)
            if current != previous[key]:
                print(step, dep.target.name, "CHANGED", previous[key], "->", current)
                previous[key] = current
            if tuple(module.weight.shape) != (module.out_features, module.in_features):
                print(step, dep.target.name, "INCONSISTENT", current)
    return previous

# Uncomment after a fresh reload/rebuild:
final_states = debug_group_prune_step_by_step(group)

## Debug the exact assignments made by Group.prune()

Run after reloading the model and rebuilding DG/group. The step-by-step call is destructive and replaces group.prune() for this run.

In [ ]:
import torch.nn as nn

def linear_state(module):
    return (module.in_features, module.out_features, tuple(module.weight.shape), id(module.weight))

def print_bridge_aliases(bridge, group):
    aliases = {
        "blocks": bridge.blocks[0].attn.q._original_component,
        "model.layers": bridge.model.layers[0].self_attn.q._original_component,
        "hf q_proj": bridge.model.layers[0].self_attn._original_component.q_proj._original_component,
        "group root": group[0][0].target.module,
    }
    for name, module in aliases.items():
        print(name, id(module), linear_state(module))
    print("all modules identical:", len({id(m) for m in aliases.values()}) == 1)
    print("all weights identical:", len({id(m.weight) for m in aliases.values()}) == 1)

def debug_group_prune_step_by_step(group):
    tracked = {id(dep.target.module): dep.target.module for dep, _ in group if isinstance(dep.target.module, nn.Linear)}
    previous = {key: linear_state(module) for key, module in tracked.items()}
    print("LINEARS BEFORE", previous)
    for step, (dep, idxs) in enumerate(group):
        print(f"\n[{step}] {dep.handler.__name__} target={dep.target.name} idxs={len(idxs)}")
        dep(idxs)
        for key, module in tracked.items():
            current = linear_state(module)
            if current != previous[key]:
                print("CHANGED:", previous[key], "->", current)
                previous[key] = current
            expected = (module.out_features, module.in_features)
            if tuple(module.weight.shape) != expected:
                print("INCONSISTENT AFTER STEP", step, dep.target.name, current)
    return previous


In [ ]:
print_bridge_aliases(bridge, group)

# DESTRUCTIVE: uncomment on a fresh model instead of group.prune().
# final_linear_states = debug_group_prune_step_by_step(group)

DONE: make correct prune_head_dims=Tru in tp and compare with analogue from transfer pruning